<a href="https://colab.research.google.com/github/gurudattamanpreet/Practice/blob/main/employee_data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import r2_score, mean_absolute_percentage_error
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler

In [2]:
df=pd.read_excel('https://github.com/gurudattamanpreet/datasets/raw/main/Employee_Data.xls')

In [3]:
df.head()

,Employee Id,First Name,Last Name,Department,Age,Experience,Salary
0,1,Joy,Bass,Sales and Marketing,28.0,3.0,32889
1,2,Sheila,Garza,Sales and Marketing,22.0,1.0,15944
2,3,John,Bryant,Customer Relations,22.0,1.0,40343
3,4,Christian,Farley,Customer Relations,22.0,1.0,19018
4,5,Colorado,Bowen,Accounting,27.0,0.0,24795


In [4]:
df.shape

(100, 7)

In [5]:
df.select_dtypes(include=['object']).nunique()

,0
First Name,97
Last Name,93
Department,3


In [6]:
df.drop(['First Name','Last Name','Employee Id'],axis=1,inplace=True)

In [7]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
Age,82.0,37.975610,9.515388,22.0,27.25,42.0,45.75,50.0
Experience,90.0,14.766667,6.889252,0.0,9.25,16.5,21.00,25.0
Salary,100.0,65066.760000,26189.874212,11830.0,48526.00,73500.5,86621.25,98180.0


In [8]:
X=df.drop(['Salary'],axis=1)
y=df['Salary']

In [9]:
X.shape

(100, 3)

In [10]:
y.shape

(100,)

In [11]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=1)

In [12]:
X_train.isna().sum()

,0
Department,15
Age,13
Experience,9


In [13]:
X_train.describe().T

,count,mean,std,min,25%,50%,75%,max
Age,67.0,37.313433,9.965408,22.0,27.0,41.0,46.0,50.0
Experience,71.0,14.436620,7.112830,0.0,8.5,16.0,21.0,25.0


In [14]:
w=['Age','Experience']
si=SimpleImputer(strategy='median')
X_train[w]=si.fit_transform(X_train[w])
X_test[w]=si.transform(X_test[w])

In [15]:
z=['Department']
si2=SimpleImputer(strategy='most_frequent')
X_train[z]=si2.fit_transform(X_train[z])
X_test[z]=si2.transform(X_test[z])

In [16]:
X_train.describe().T

,count,mean,std,min,25%,50%,75%,max
Age,80.0,37.9125,9.210882,22.0,28.75,41.0,45.0,50.0
Experience,80.0,14.6125,6.713851,0.0,10.00,16.0,20.0,25.0


In [17]:
r=X_train[['Age','Experience']]
Q1=r.quantile(0.25)
Q3=r.quantile(0.75)
IQR=Q3-Q1
lower_bound=Q1-(1.5*IQR)
upper_bound=Q3+(1.5*IQR)
lower_outlier=r<lower_bound
upper_outlier=r>upper_bound

In [18]:
print(lower_outlier.sum())

Age           0
Experience    0
dtype: int64


In [19]:
print(upper_outlier.sum())

Age           0
Experience    0
dtype: int64


In [20]:
X_train['Department'].value_counts()

,count
Department,
Sales and Marketing,40
Customer Relations,22
Accounting,18


In [21]:
# u=['Department']
# X_train_scaled=pd.get_dummies(X_train,columns=u,drop_first=True)
# X_test_scaled=pd.get_dummies(X_test,columns=u,drop_first=True)
# X_test_scaled=X_test.reindex(columns=X_test.columns,fill_value=0)

In [22]:
u = ['Department']
ohe = OneHotEncoder(drop='first', handle_unknown='ignore')

ct = ColumnTransformer(transformers=[('ohe', ohe, u)], remainder='passthrough')

X_train_scaled = ct.fit_transform(X_train)
X_test_scaled = ct.transform(X_test)

In [23]:
sc = StandardScaler()
X_train_encoded = sc.fit_transform(X_train_scaled)
X_test_encoded = sc.transform(X_test_scaled)

In [24]:
lr=LinearRegression()
lr.fit(X_train_encoded,y_train)

LinearRegression()

In [25]:
y_train_pred=lr.predict(X_train_encoded)
y_test_pred=lr.predict(X_test_encoded)

In [26]:
r2_score(y_test,y_test_pred)

0.7131266628906883

In [27]:
r2_score(y_train,y_train_pred)

0.841160796327261

In [28]:
mean_absolute_percentage_error(y_test,y_test_pred)

0.1842994817494168

In [29]:
mean_absolute_percentage_error(y_train,y_train_pred)

0.21746081809904444